# ⚡ SoulX-FlashHead + UniTalk Adapter — RunPod Inference

Generates a talking-head video from a **reference image + audio file** using the
**UniTalk MoshiToWav2VecAdapter** instead of Wav2Vec2 for audio encoding.

**Audio pipeline comparison:**

| Standard SoulX | This notebook (Adapter mode) |
|:---|:---|
| WAV → Wav2Vec2 CNN → 12-layer transformer → `[T, 12, 768]` | WAV → Moshi Mimi codec → LM → `[T, 4096]` → Adapter (12-layer transformer) → `[T, 12, 768]` |

The adapter output is **drop-in compatible** with SoulX's AudioProjModel, so the
DiT denoising and VAE decoding stages are unchanged.

### Steps
1. Check GPU / environment  
2. Clone repository  
3. Install dependencies  
4. Download model weights (SoulX + wav2vec2 + Moshi)  
5. Place your adapter checkpoint  
6. Run inference  
7. Preview the generated video  

---
### GPU guidance

| Model | Moshi LM | Adapter | DiT+VAE | Minimum VRAM |
|:--|:--|:--|:--|:--|
| **Lite** | ~14 GB (bf16) | ~0.4 GB | ~8 GB | 24 GB (A10G / A100 40GB) |
| **Pro** | ~14 GB (bf16) | ~0.4 GB | ~12 GB | 40 GB (A100 40GB / H100) |

> **Tip:** On RunPod, use an **A100 40GB** pod for Lite; an **A100 80GB or H100** for Pro.

## 1. Check GPU & environment

In [ ]:
!nvidia-smi

In [ ]:
import sys, platform
print('Python  :', sys.version.split()[0])
print('Platform:', platform.platform())
try:
    import torch
    print('torch   :', torch.__version__)
    print('CUDA    :', torch.version.cuda)
    print('GPUs    :', torch.cuda.device_count())
    for i in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(i)
        print(f'  GPU {i}  {props.name}  {props.total_memory / 1e9:.1f} GB')
except ImportError:
    print('torch   : not installed yet (will be installed below)')

## 2. Clone / locate the repository

We work inside `/workspace` (persistent volume on RunPod).  
If you already cloned the `merge-soul-n-uni` project, adjust `REPO_DIR` accordingly.

In [ ]:
import os

WORKSPACE = '/workspace' if os.path.isdir('/workspace') else os.path.expanduser('~')

# ── Option A: the merged project was uploaded / already lives here ──────────
# REPO_DIR = os.path.join(WORKSPACE, 'merge-soul-n-uni')

# ── Option B: clone from your own fork / private repo ───────────────────────
REPO_DIR = os.path.join(WORKSPACE, 'merge-soul-n-uni')
if not os.path.isdir(REPO_DIR):
    !git clone https://github.com/MoshiHead/merge-soul-n-uni.git {REPO_DIR}

# ── SoulX subfolder (contains flash_head/, generate_video_adapter.py, etc.) ─
SOULX_DIR = os.path.join(REPO_DIR, 'SoulX-FlashHead-original')

os.chdir(SOULX_DIR)
print('Working directory:', os.getcwd())
!ls -la

## 3. Install dependencies

### 3a. PyTorch (CUDA 12.8 build)

In [ ]:
!pip install torch==2.7.1 torchvision==0.22.1 --index-url https://download.pytorch.org/whl/cu128

### 3b. SoulX-FlashHead requirements

In [ ]:
# Fix mediapipe pin and remove nvidia-nccl (not needed on single-GPU pods)
!sed -e 's/mediapipe==0.10.9/mediapipe>=0.10.13/' \
     -e '/nvidia-nccl-cu12/d' \
     requirements.txt > requirements_py312.txt
!pip install -r requirements_py312.txt --ignore-installed blinker
!python -c "import imageio, librosa, cv2, mediapipe; print('core deps OK')"

### 3c. Pin transformers version (required by SoulX)

In [ ]:
!pip install 'transformers==4.57.3' 'huggingface-hub>=0.34.0,<1.0' --no-deps
!python -c "import transformers; print('transformers', transformers.__version__)"

### 3d. FlashAttention (prebuilt wheel — much faster than building from source)

In [ ]:
!pip install ninja
# Prebuilt wheel for torch 2.7 / cu12 / Python 3.10 — adjust if your stack differs
!pip install flash_attn==2.8.0.post2 --no-build-isolation

### 3e. Moshi (for offline audio encoding)

The Moshi package is used only for its **Mimi codec + LM forward pass** to convert
the input WAV into 4096-D tokens that the adapter can process.

In [ ]:
!pip install moshi sentencepiece sphn
!python -c "import moshi; print('moshi OK, version:', moshi.__version__ if hasattr(moshi, '__version__') else 'installed')"

### 3f. FFmpeg

In [ ]:
!ffmpeg -version >/dev/null 2>&1 && echo 'ffmpeg already installed' \
    || (apt-get update -y && apt-get install -y ffmpeg)
!ffmpeg -version | head -n 1

## 4. Download model weights

Three components are needed:

| Component | Repo | Size | Where used |
|:--|:--|:--|:--|
| SoulX-FlashHead 1.3B | `Soul-AILab/SoulX-FlashHead-1_3B` | ~14 GB | DiT + VAE |
| wav2vec2-base-960h | `facebook/wav2vec2-base-960h` | ~1.1 GB | Loaded by pipeline (not used for audio encoding in adapter mode) |
| Moshi Helium | downloaded automatically by `CheckpointInfo` | ~14 GB | Audio encoding |

> **Moshi** is fetched automatically when `WavToMoshiTokens.load()` is called (step 6).  
> You only need to download SoulX and wav2vec2 here.

In [ ]:
import os
# os.environ['HF_ENDPOINT'] = 'https://hf-mirror.com'   # uncomment if in mainland China

!pip install -q -U huggingface_hub

!hf download Soul-AILab/SoulX-FlashHead-1_3B --local-dir ./models/SoulX-FlashHead-1_3B
!hf download facebook/wav2vec2-base-960h     --local-dir ./models/wav2vec2-base-960h

print('\nModels directory:')
!ls -R ./models | head -n 40

## 5. Place your adapter checkpoint

Copy (or symlink) your trained `.pt` file to `checkpoints/`.  
The default path expected by `generate_video_adapter.py` is:
```
<project_root>/checkpoints/moshi_to_adapter/moshi_to_flashhead_phase1_best.pt
```

Use the cell below to upload or copy your checkpoint.

In [ ]:
import os

REPO_DIR  = os.path.dirname(os.getcwd())   # merge-soul-n-uni/
CKPT_DIR  = os.path.join(REPO_DIR, 'checkpoints', 'moshi_to_adapter')
os.makedirs(CKPT_DIR, exist_ok=True)

# ── Option A: copy from a mounted volume / local path ───────────────────────
# !cp /path/to/your/moshi_to_flashhead_phase1_best.pt {CKPT_DIR}/

# ── Option B: download from your own storage ────────────────────────────────
# !wget -P {CKPT_DIR} https://YOUR_STORAGE/moshi_to_flashhead_phase1_best.pt

# ── Option C (testing only): use random weights ──────────────────────────────
# The script will warn you and continue with random adapter weights.
# Video will animate but NOT lip-sync until a trained checkpoint is provided.

ADAPTER_CKPT = os.path.join(CKPT_DIR, 'moshi_to_flashhead_phase1_best.pt')

if os.path.isfile(ADAPTER_CKPT):
    size_mb = os.path.getsize(ADAPTER_CKPT) / 1e6
    print(f'Adapter checkpoint found: {ADAPTER_CKPT}  ({size_mb:.1f} MB)')
else:
    print(f'WARNING: Adapter checkpoint not found at {ADAPTER_CKPT}')
    print('The inference will run with RANDOM adapter weights (no lip-sync).')
    print('Place your trained checkpoint at the path above, or change ADAPTER_CKPT below.')

print('\nAdapter checkpoint path:', ADAPTER_CKPT)

## 6. Run inference

Edit the paths below to point to your own image and audio.  
The bundled examples from the SoulX repo work out of the box.

### 6a. Lite model — single GPU (recommended starting point)

In [ ]:
import os

REPO_DIR     = os.path.dirname(os.getcwd())    # merge-soul-n-uni/
SOULX_DIR    = os.getcwd()                     # SoulX-FlashHead-original/

CKPT_DIR     = os.path.join(SOULX_DIR, 'models', 'SoulX-FlashHead-1_3B')
WAV2VEC_DIR  = os.path.join(SOULX_DIR, 'models', 'wav2vec2-base-960h')
ADAPTER_CKPT = os.path.join(REPO_DIR,  'checkpoints', 'moshi_to_adapter',
                             'moshi_to_flashhead_phase1_best.pt')
MOSHI_PKG    = os.path.join(REPO_DIR,  'moshi', 'moshi')   # adjust if different

# ── Your inputs ─────────────────────────────────────────────────────────────
COND_IMAGE   = os.path.join(SOULX_DIR, 'examples', 'girl.png')
AUDIO_PATH   = os.path.join(SOULX_DIR, 'examples', 'podcast_sichuan_16k.wav')

# ── Uncomment to use your own files ─────────────────────────────────────────
# COND_IMAGE = '/path/to/your/portrait.png'
# AUDIO_PATH = '/path/to/your/speech.wav'

print('Checkpoint dir :', CKPT_DIR)
print('wav2vec2 dir   :', WAV2VEC_DIR)
print('Adapter ckpt   :', ADAPTER_CKPT)
print('Moshi pkg      :', MOSHI_PKG)
print('Input image    :', COND_IMAGE)
print('Input audio    :', AUDIO_PATH)

In [ ]:
# Run adapter-based inference (Lite model, single GPU)
!python generate_video_adapter.py \
    --ckpt_dir      "{CKPT_DIR}" \
    --wav2vec_dir   "{WAV2VEC_DIR}" \
    --model_type    lite \
    --cond_image    "{COND_IMAGE}" \
    --audio_path    "{AUDIO_PATH}" \
    --adapter_ckpt  "{ADAPTER_CKPT}" \
    --moshi_pkg     "{MOSHI_PKG}"

### 6b. Pro model — single GPU

Higher quality, ~600ms per chunk.  Requires ≥40 GB VRAM when Moshi is also loaded.

In [ ]:
!python generate_video_adapter.py \
    --ckpt_dir      "{CKPT_DIR}" \
    --wav2vec_dir   "{WAV2VEC_DIR}" \
    --model_type    pro \
    --cond_image    "{COND_IMAGE}" \
    --audio_path    "{AUDIO_PATH}" \
    --adapter_ckpt  "{ADAPTER_CKPT}" \
    --moshi_pkg     "{MOSHI_PKG}"

### 6c. Custom save path + Phase 2 adapter checkpoint

If you have a Phase 2 checkpoint (higher lip-sync quality), point `--adapter_ckpt` at it.

In [ ]:
ADAPTER_CKPT_PHASE2 = os.path.join(
    REPO_DIR, 'checkpoints', 'moshi_to_adapter',
    'moshi_to_flashhead_phase2_best.pt'
)
OUTPUT_FILE = '/workspace/my_output_video.mp4'

# Only run if the Phase 2 checkpoint actually exists
if os.path.isfile(ADAPTER_CKPT_PHASE2):
    get_ipython().system(
        f'python generate_video_adapter.py '
        f'--ckpt_dir "{CKPT_DIR}" '
        f'--wav2vec_dir "{WAV2VEC_DIR}" '
        f'--model_type lite '
        f'--cond_image "{COND_IMAGE}" '
        f'--audio_path "{AUDIO_PATH}" '
        f'--adapter_ckpt "{ADAPTER_CKPT_PHASE2}" '
        f'--moshi_pkg "{MOSHI_PKG}" '
        f'--save_file "{OUTPUT_FILE}"'
    )
else:
    print(f'Phase 2 checkpoint not found at {ADAPTER_CKPT_PHASE2}  — skipping.')

## 7. Preview the generated video

In [ ]:
import glob, os
from IPython.display import Video, display

# Find the most recently created adapter-result MP4
mp4s = sorted(
    [p for p in glob.glob('**/*.mp4', recursive=True)
     if 'adapter' in os.path.basename(p)],
    key=lambda p: os.path.getmtime(p),
    reverse=True,
)

# Fall back to any MP4 if no adapter-named file found yet
if not mp4s:
    mp4s = sorted(
        glob.glob('**/*.mp4', recursive=True),
        key=lambda p: os.path.getmtime(p),
        reverse=True,
    )

if mp4s:
    latest = mp4s[0]
    print('Most recent video:', os.path.abspath(latest))
    display(Video(latest, embed=True, width=512))
else:
    print('No .mp4 found yet — run an inference cell above first.')

## 8. Use Python API directly (advanced)

For custom scripting without the CLI, you can call each stage directly.
This is useful for batch processing multiple images or audio clips.

In [ ]:
# ── Example: batch-generate videos for a list of images ─────────────────────

import sys, os, math, torch
from collections import deque
from loguru import logger

REPO_DIR  = os.path.dirname(os.getcwd())
SOULX_DIR = os.getcwd()

for _p in [SOULX_DIR, REPO_DIR]:
    if _p not in sys.path:
        sys.path.insert(0, _p)

# ── Load pipeline (once) ─────────────────────────────────────────────────────
from flash_head.inference import get_pipeline, get_base_data, get_infer_params, run_pipeline
from unitalk.models.adapter import MoshiToWav2VecAdapter
from unitalk.settings import MOSHI_DIM, WAV2VEC_DIM, WAV2VEC_LAYERS, DEQUE_SIZE
from unitalk.offline_encoder import WavToMoshiTokens
from unitalk.streaming import get_audio_embedding_from_tokens

DEVICE = 'cuda'

pipeline = get_pipeline(
    world_size=1,
    ckpt_dir=os.path.join(SOULX_DIR, 'models', 'SoulX-FlashHead-1_3B'),
    model_type='lite',
    wav2vec_dir=os.path.join(SOULX_DIR, 'models', 'wav2vec2-base-960h'),
)
infer_params      = get_infer_params()
frame_num         = infer_params['frame_num']
motion_frames_num = infer_params['motion_frames_num']
slice_len         = frame_num - motion_frames_num
tgt_fps           = infer_params['tgt_fps']
tokens_per_chunk  = slice_len // 2

adapter = MoshiToWav2VecAdapter(moshi_dim=MOSHI_DIM, hidden_dim=WAV2VEC_DIM, num_layers=WAV2VEC_LAYERS)
adapter.load_checkpoint(
    os.path.join(REPO_DIR, 'checkpoints', 'moshi_to_adapter', 'moshi_to_flashhead_phase1_best.pt')
)
adapter = adapter.to(DEVICE).eval()

encoder = WavToMoshiTokens(
    moshi_pkg=os.path.join(REPO_DIR, 'moshi', 'moshi'),
    hf_repo='kyutai/moshiko-pytorch-bf16',
    device=DEVICE,
)
encoder.load()

print('All models loaded.')

In [ ]:
def generate_video_from_image_and_audio(
    image_path: str,
    audio_path: str,
    output_path: str,
    seed: int = 42,
) -> str:
    """High-level helper: image + audio → MP4 using the loaded models."""
    import imageio, subprocess, numpy as np

    get_base_data(pipeline, cond_image_path_or_dir=image_path,
                  base_seed=seed, use_face_crop=False)

    all_tokens = encoder.encode(audio_path)        # [N, 4096] CPU
    N_total = all_tokens.shape[0]
    n_chunks = max(1, math.ceil(N_total / tokens_per_chunk))

    token_deque = deque(maxlen=DEQUE_SIZE)
    for _ in range(DEQUE_SIZE):
        token_deque.append(torch.zeros(MOSHI_DIM))

    frames_all = []
    with torch.inference_mode():
        for chunk_idx in range(n_chunks):
            start = chunk_idx * tokens_per_chunk
            end   = min(start + tokens_per_chunk, N_total)
            for tok in all_tokens[start:end]:
                token_deque.append(tok)
            for _ in range(tokens_per_chunk - (end - start)):
                token_deque.append(torch.zeros(MOSHI_DIM))

            snap = torch.stack(list(token_deque), dim=0)
            audio_emb = get_audio_embedding_from_tokens(
                adapter, snap, frame_num,
                device=torch.device(DEVICE),
                dtype=pipeline.param_dtype,
            )
            video = run_pipeline(pipeline, audio_emb)
            frames_all.append(video[motion_frames_num:].cpu())

    tmp = output_path.replace('.mp4', '_tmp.mp4')
    with imageio.get_writer(tmp, format='mp4', mode='I', fps=tgt_fps,
                             codec='h264', ffmpeg_params=['-bf', '0']) as w:
        for frames in frames_all:
            for i in range(frames.shape[0]):
                w.append_data(frames[i].numpy().astype(np.uint8))
    subprocess.run(['ffmpeg', '-i', tmp, '-i', audio_path,
                    '-c:v', 'copy', '-c:a', 'aac', '-shortest',
                    output_path, '-y'], check=True)
    os.remove(tmp)
    return output_path


# ── Example usage ────────────────────────────────────────────────────────────
out = generate_video_from_image_and_audio(
    image_path='examples/girl.png',
    audio_path='examples/podcast_sichuan_16k.wav',
    output_path='sample_results/api_result.mp4',
)
print('Saved to:', out)

## Troubleshooting

| Problem | Fix |
|:---|:---|
| `CUDA out of memory` | Use a larger-VRAM pod; or free Moshi from memory after encoding (`del encoder; torch.cuda.empty_cache()`) before running the SoulX pipeline |
| `No module named 'moshi'` | `pip install moshi sentencepiece sphn` |
| Adapter checkpoint not found | Copy your `.pt` file to `checkpoints/moshi_to_adapter/` or pass the exact path with `--adapter_ckpt` |
| Video animates but no lip-sync | You are using random adapter weights — a trained checkpoint is required |
| `CheckpointInfo.from_hf_repo` fails | Set `HF_ENDPOINT=https://hf-mirror.com` in mainland China, or authenticate with `hf auth login` |
| Slow first chunk | Normal — DiT is JIT-compiled on first run; subsequent chunks are fast |
| `flash_head/configs/infer_params.yaml` not found | Make sure `os.chdir(SOULX_DIR)` ran before importing `flash_head` |

---
*This notebook is part of the `merge-soul-n-uni` project combining SoulX-FlashHead + UniTalk.*